# Sigma0 Stability Analysis
Mirrors the sigma1/sigma2 stability notebooks but for **sigma0** (ISI=0).

| Section | Question |
|---------|----------|
| A | Per-experiment-count stability at ISI=0 |
| B | k_stimuli sensitivity |

In [ ]:
import sys, os, yaml, torch, random
import matplotlib.pyplot as plt, numpy as np, pandas as pd
from collections import defaultdict
from scipy.stats import norm
from sklearn.metrics import roc_auc_score
from pathlib import Path
from scipy.spatial.distance import pdist
from tqdm.notebook import trange, tqdm

sys.path.append('/om2/user/jmhicks/projects/TextureStreaming/code/')
sys.path.append('/om2/user/bjmedina/auditory-memory/memory/utls/')
sys.path.append('/om2/user/bjmedina/auditory-memory/memory/src/model/')
sys.path.append('/om2/user/bjmedina/auditory-memory/memory/')

from chexture_choolbox.auditorytexture.texture_model import TextureModel
from chexture_choolbox.auditorytexture.helpers import FlattenStats
from texture_prior.params import model_params, statistics_set, texture_dataset
from texture_prior.utils import path
from utls.plotting import ensure_dir
from utls.loading import (load_results_with_exclusion_2, move_sequences_to_used,
                           load_results_with_exclusion_no_dropping)
from utls.runners_v2 import run_experiment_scores, make_noise_schedule
from utls.runners_utils import *
from encoders import *
from utls.toy_experiments import (
    make_isi_n_block_experiment, make_toy_experiment_list, make_multi_isi_toy_experiments,
)
from utls.sigma_fitting import log_mid, make_grid, auc_to_dprime

In [ ]:
def load_config(cfg_path):
    cfg_path = Path(cfg_path)
    if not cfg_path.exists():
        raise FileNotFoundError(cfg_path)
    with open(cfg_path) as f:
        return yaml.safe_load(f), cfg_path

def median_pairwise_distance(X, metric="euclidean", n_samples=500, seed=0):
    rng = np.random.default_rng(seed)
    idx = rng.choice(X.shape[0], size=min(n_samples, X.shape[0]), replace=False)
    return float(np.median(pdist(X[idx], metric=metric)))

CONFIG_PATH = (
    "/om2/user/bjmedina/auditory-memory/memory/"
    "model_yamls/three-regime/resnet50/nontime_avg/run_000005.yaml"
)
model_cfg, model_cfg_path = load_config(CONFIG_PATH)
print(model_cfg)

In [ ]:
exp_cfg    = model_cfg["experiment"]
which_task = exp_cfg["which_task"]
is_multi   = exp_cfg["is_multi"]
which_isi  = exp_cfg.get("which_isi", None)
isis       = [0, 1, 2, 4, 8, 16, 32, 64] if is_multi else [0, which_isi]
metric     = model_cfg["metric"]
noise_cfg  = model_cfg["noise_model"]
noise_mode = noise_cfg["name"]
t_step     = noise_cfg["t_step"]
repr_cfg   = model_cfg["representation"]
time_avg   = repr_cfg["time_avg"]
encoder_type = repr_cfg["type"]
layer      = repr_cfg.get("layer", None)
pc_dims    = repr_cfg.get("pc_dims", None)

exp_list, all_files, name_to_idx, human_runs, task_name, hr_task_name = (
    load_experiment_data(which_task, which_isi, is_multi, old=False)
)
human_curve  = compute_human_curve(human_runs, is_multi, which_isi)
time_avg_tag = "time_avg" if time_avg else "nontime_avg"
print("ISIs:", isis)
print("Human d':", human_curve)

In [ ]:
NN_ENCODERS  = {"kell2018", "resnet50"}
encoder_task = "word_speaker_audioset" if encoder_type in NN_ENCODERS else "audioset"
encoder_cfg  = dict(
    encoder_type=encoder_type, model_name=encoder_type, task=encoder_task,
    statistics_dict=statistics_set.statistics, model_params=model_params,
    pc_dims=pc_dims, sr=20000, duration=2.0, rms_level=0.05,
    time_avg=time_avg, device="cuda",
)
if encoder_type in NN_ENCODERS: encoder_cfg["layer"] = layer
if encoder_type == "texture":   encoder_cfg["pc_dims"] = pc_dims

encoder_name = make_encoder_name(encoder_cfg)
encoder      = build_encoder(encoder_cfg)
X            = encode_stimuli(encoder, all_files)
X_np         = X.detach().cpu().numpy()
print("Shape:", X_np.shape, " NaN?", torch.isnan(X).any().item())

d50 = median_pairwise_distance(X_np, metric="cosine")
print(f"d50 = {d50:.6f}")

param_bounds = {
    "sigma0": (noise_cfg["sigma0_min"],         noise_cfg["sigma0_max"]),
    "sigma1": (noise_cfg["sigma1_min"] * d50,   noise_cfg["sigma1_max"] * d50),
    "sigma2": (noise_cfg["sigma2_min"] * d50,   noise_cfg["sigma2_max"] * d50),
}
for k, v in param_bounds.items():
    print(f"  {k}: ({v[0]:.6f}, {v[1]:.6f})")

stimulus_pool = sorted({s for seq in exp_list for s in seq})
print(f"Stimulus pool: {len(stimulus_pool)}")

In [ ]:
# ---- fix other sigmas at geometric means ----
sigma1_fixed = log_mid(*param_bounds['sigma1'])
sigma2_fixed = log_mid(*param_bounds['sigma2'])
print(f'Fixed sigma1 = {sigma1_fixed:.6f}')
print(f'Fixed sigma2 = {sigma2_fixed:.6f}')

N_MC      = 64
N_PER_DIM = 8
K_STIM    = 10   # ISI=0 only needs k>=1, but 10 is consistent with sigma1

# param_bounds['sigma0'] = (3.0, 0.5) in the YAML — sort to ensure lo < hi
sigma0_lo = min(param_bounds['sigma0'])
sigma0_hi = max(param_bounds['sigma0'])
sigma0_grid = make_grid(sigma0_lo, sigma0_hi, N_PER_DIM, spacing='log')
print('sigma0 grid:', [f'{v:.5f}' for v in sigma0_grid])

isi_to_hc_idx = {isi_val: i for i, isi_val in enumerate(isis)}
print(f"\nHuman d' at ISI=0: {human_curve[isi_to_hc_idx[0]]:.4f}")

In [ ]:
def run_sigma_sweep(sigma_name, sigma_grid, fixed_sigmas, exps_by_isi,
                    isi_to_hc_idx, human_curve, N_MC, t_step, noise_mode,
                    metric, X, name_to_idx, base_seed=0):
    """Sweep one sigma over sigma_grid; for each value run N_MC reps.
    Returns list of dicts with sigma_value, mse_mean, mse_std, dprime_mean, dprime_std.
    """
    results = []
    for sig_idx, sigma_val in enumerate(sigma_grid):
        sigmas = dict(fixed_sigmas)
        sigmas[sigma_name] = sigma_val
        mse_per_rep    = []
        dprime_per_rep = []

        for rep in trange(N_MC, desc=f"{sigma_name}={sigma_val:.4g}", leave=False):
            rep_mse = []
            rep_dp  = []
            for isi_val, exps in exps_by_isi.items():
                if not exps: continue
                hc_idx   = isi_to_hc_idx.get(isi_val)
                if hc_idx is None: continue
                human_dp = human_curve[hc_idx]
                run_out  = run_experiment_scores(
                    sigma0=sigmas["sigma0"], sigma1=sigmas["sigma1"], sigma2=sigmas["sigma2"],
                    t_step=t_step, rate=0, noise_mode=noise_mode,
                    metric=metric, X0=X, name_to_idx=name_to_idx,
                    experiment_list=exps, debug=False,
                    seed=base_seed + isi_val*1_000_000 + sig_idx*10_000 + rep,
                )
                hits = np.asarray(run_out["hits"]); fas = np.asarray(run_out["fas"])
                if len(hits)==0 or len(fas)==0: continue
                y = np.concatenate([np.ones(len(hits)), np.zeros(len(fas))])
                dp = auc_to_dprime(roc_auc_score(y, -np.concatenate([hits,fas])))
                rep_mse.append((dp - human_dp)**2)
                rep_dp.append(dp)
            if rep_mse:
                mse_per_rep.append(np.mean(rep_mse))
                dprime_per_rep.append(np.mean(rep_dp))

        results.append({
            "sigma_value":  sigma_val,
            "mse_mean":     np.mean(mse_per_rep)    if mse_per_rep    else np.nan,
            "mse_std":      np.std(mse_per_rep)     if mse_per_rep    else np.nan,
            "dprime_mean":  np.mean(dprime_per_rep) if dprime_per_rep else np.nan,
            "dprime_std":   np.std(dprime_per_rep)  if dprime_per_rep else np.nan,
        })
    return results

## Section A: Per-Experiment-Count Stability

ISI=0 only. Vary n_experiments in [20, 40, 80, 160].

In [ ]:
EXP_COUNTS = [20, 40, 80, 160]
per_nexp_results = {}

hc_idx = isi_to_hc_idx[0]
print(f"=== ISI=0  human d' = {human_curve[hc_idx]:.4f} ===")

for n_exp in EXP_COUNTS:
    exps = make_toy_experiment_list(
        stimulus_pool, isi=0, n_experiments=n_exp,
        k_stimuli=K_STIM, seed=n_exp)
    print(f'  n_exp={n_exp}: {len(exps)} exps, '
          f'avg len {np.mean([len(e) for e in exps]):.1f}')

    res = run_sigma_sweep(
        sigma_name='sigma0', sigma_grid=sigma0_grid,
        fixed_sigmas={'sigma0': 0, 'sigma1': sigma1_fixed, 'sigma2': sigma2_fixed},
        exps_by_isi={0: exps},
        isi_to_hc_idx=isi_to_hc_idx, human_curve=human_curve,
        N_MC=N_MC, t_step=t_step, noise_mode=noise_mode,
        metric=metric, X=X, name_to_idx=name_to_idx,
        base_seed=n_exp * 1_000_000,
    )
    per_nexp_results[n_exp] = res

print('Done.')

In [ ]:
hc_idx = isi_to_hc_idx[0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# MSE panel
ax = axes[0]
for n_exp, res in per_nexp_results.items():
    df = pd.DataFrame(res)
    ax.plot(df.sigma_value, df.mse_mean, 'o-', label=f'N={n_exp}')
    ax.fill_between(df.sigma_value, df.mse_mean - df.mse_std,
                    df.mse_mean + df.mse_std, alpha=0.2)
ax.set_xscale('log')
ax.set_xlabel(r'$\sigma_0$')
ax.set_ylabel('MSE')
ax.set_title(f'ISI=0  (human d\'={human_curve[hc_idx]:.2f})')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# d' panel
ax = axes[1]
for n_exp, res in per_nexp_results.items():
    df = pd.DataFrame(res)
    ax.errorbar(df.sigma_value, df.dprime_mean, yerr=df.dprime_std,
                fmt='o-', capsize=3, label=f'N={n_exp}')
ax.axhline(human_curve[hc_idx], color='k', ls='--', label='Human', alpha=0.6)
ax.set_xscale('log'); ax.set_xlabel(r'$\sigma_0$'); ax.set_ylabel("d'")
ax.set_title('ISI=0'); ax.legend(fontsize=8); ax.grid(alpha=0.3)

fig.suptitle(f'Sigma0 per-experiment-count stability  ({encoder_type}-{layer}-{time_avg_tag})  N_MC={N_MC}', y=1.03)
plt.tight_layout(); plt.show()

## Section B: k_stimuli Sensitivity

Fixed n_exp=40. Test how the number of distinct stimuli per experiment affects the MSE landscape.

In [ ]:
K_VALUES    = [5, 10, 20, 40]
N_EXP_FIXED = 40
k_results   = {}

for k_stim in K_VALUES:
    exps = make_toy_experiment_list(
        stimulus_pool, isi=0, n_experiments=N_EXP_FIXED,
        k_stimuli=k_stim, seed=k_stim * 7)
    print(f'k_stim={k_stim}: {len(exps)} exps, '
          f'avg len {np.mean([len(e) for e in exps]):.1f}')
    res = run_sigma_sweep(
        sigma_name='sigma0', sigma_grid=sigma0_grid,
        fixed_sigmas={'sigma0': 0, 'sigma1': sigma1_fixed, 'sigma2': sigma2_fixed},
        exps_by_isi={0: exps},
        isi_to_hc_idx=isi_to_hc_idx, human_curve=human_curve,
        N_MC=N_MC, t_step=t_step, noise_mode=noise_mode,
        metric=metric, X=X, name_to_idx=name_to_idx,
        base_seed=900_000_000 + k_stim * 1_000_000,
    )
    k_results[k_stim] = res

print('Done.')

In [ ]:
plt.figure(figsize=(8, 5))
for k_stim, res in k_results.items():
    df = pd.DataFrame(res)
    plt.plot(df.sigma_value, df.mse_mean, 'o-', label=f'k={k_stim}')
    plt.fill_between(df.sigma_value, df.mse_mean - df.mse_std,
                     df.mse_mean + df.mse_std, alpha=0.2)
plt.xscale('log'); plt.xlabel(r'$\sigma_0$'); plt.ylabel('MSE')
plt.title(f'Sigma0 k_stimuli sensitivity  n_exp={N_EXP_FIXED}, N_MC={N_MC}')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## Summary

*(Fill in after running)*

**Recommended Stage A settings:**

In [ ]:
print('Cost estimates (run_experiment_scores calls per fit):')
print(f'  Current: n_grid=15, n_mc=32, n_refine=2, 1 ISI = {15*32*1*2}')
for n_mc in [16, 32]:
    for n_grid in [8, 10, 15]:
        print(f'  n_grid={n_grid} n_mc={n_mc}: {n_grid*n_mc*1*2}')